In [16]:
import requests, sys
sys.path.append('backend/app/')
import pandas as pd, geopandas as gpd
from backend.app.services import functions

In [17]:
CLIENT_ID = "f6d537e9-cf9d-4a87-a838-02d607fe68bd"
url = "https://frost.met.no"
url_station = f"{url}/observations/availableTimeSeries/v0.jsonld"
station_ids = [ "SN61065", "SN60670", "SN60400"]
params = {
    "sources": ",".join(station_ids)
}

In [18]:
response = requests.get(url_station, params=params, auth=(CLIENT_ID, ""))
response.raise_for_status()
stations = response.json()
df = pd.DataFrame(stations['data'])

In [19]:
df.head()

,sourceId,level,validFrom,timeOffset,timeResolution,timeSeriesId,elementId,unit,performanceCategory,exposureCategory,status,uri,validTo,codeTable
0,SN60400:0,"{'levelType': 'height_above_ground', 'unit': '...",2022-09-01T00:00:00.000Z,PT18H,P3M,0,min(air_temperature P3M),degC,C,1,Authoritative,https://frost.met.no/observations/v0.jsonld?so...,NaN,NaN
1,SN60670:0,NaN,2016-10-26T00:00:00.000Z,PT0H,P1D,0,mean(dew_point_temperature P1D),degC,C,4,Authoritative,https://frost.met.no/observations/v0.jsonld?so...,NaN,NaN
2,SN60400:0,NaN,2022-08-05T00:00:00.000Z,PT0H,PT10M,0,sum(precipitation_amount PT10M),mm,C,2,Authoritative,https://frost.met.no/observations/v0.jsonld?so...,NaN,NaN
3,SN60400:0,NaN,1895-07-01T00:00:00.000Z,PT6H,P1D,0,sum(precipitation_amount P1D),mm,C,2,Authoritative,https://frost.met.no/observations/v0.jsonld?so...,NaN,NaN
4,SN60400:0,NaN,2022-08-06T00:00:00.000Z,PT18H,P1D,0,sum(precipitation_amount P1D),mm,C,2,Authoritative,https://frost.met.no/observations/v0.jsonld?so...,NaN,NaN


In [6]:
url_series = f"{url}/Parameters"
response = requests.get(
    url_series,
    headers=headers,
    timeout=30
)
response.raise_for_status()
series_data = response.json()
df = pd.DataFrame(series_data['data'])
df

,parameter,parameterName,parameterNameEng,unit
0,0,Nedbør,Precipitation,mm
1,1000,Vannstand,Stage,m
2,1001,Vannføring,Discharge,m³/s
3,1002,Vannhastighet,Water speed,m/s
4,1003,Vanntemperatur,Water temperature,°C
5,1004,Magasinvolum,Reservoir volume,millioner m³
6,1006,Ledningsevne,Conductivity,µS/cm
7,1007,pH,pH,pH
8,1012,Overvannstand kraftverk,Head water level-hydropow.,m
9,1055,Driftsvannføring,Operating discharge,m³/s


In [ ]:
url_series = f"{url}/Series"
stations = ['1.15.0', '1.200.0', '1.42.0']
pr = set()
for st in stations:
    params = {
        'StationId': st
    }
    response = requests.get(
        url_series,
        headers=headers,
        params=params,
        timeout=30
    )
    response.raise_for_status()
    data = response.json()
    records = data.get('data', [])
    df = pd.DataFrame(records)
    for i in df['parameter'].values:
        pr.add(i)
    print(df)
print(pr)

  stationId stationName parameterName  parameter  versionNo  methodKey  \
0    1.15.0      Femsjø     Vannstand       1000          2          3   
1    1.15.0      Femsjø  Magasinvolum       1004          2          3   

           unit            serieFrom serieTo  latitude  longitude  \
0             m  1939-07-01T00:00:00    None  59.13015   11.48516   
1  millioner m³                  NaN    None  59.13015   11.48516   

   utmEast_Z33  utmNorth_Z33  masl councilNumber councilName countyName  \
0       298887       6559841    81          3101      Halden    Østfold   
1       298887       6559841    81          3101      Halden    Østfold   

  observationPlace measuredOrDerived  \
0          Magasin              Målt   
1           Ukjent           Avledet   

                                      resolutionList  
0  [{'resTime': 0, 'method': 'Instantaneous', 'ti...  
1  [{'resTime': 0, 'method': 'Instantaneous', 'ti...  
set()


In [12]:
df

,Time,stationId,1000,1004
0,2026-09-13 12:00:00+00:00,1.15.0,79.116,NaN
1,2026-09-13 13:00:00+00:00,1.15.0,79.118,NaN
2,2026-09-13 14:00:00+00:00,1.15.0,79.116,NaN
3,2026-09-13 15:00:00+00:00,1.15.0,79.118,NaN
4,2026-09-13 16:00:00+00:00,1.15.0,79.117,NaN
...,...,...,...,...
2400,2026-10-03 08:00:00+00:00,1.42.0,NaN,7.692012
2401,2026-10-03 09:00:00+00:00,1.42.0,NaN,7.809049
2402,2026-10-03 10:00:00+00:00,1.42.0,NaN,7.666026
2403,2026-10-03 11:00:00+00:00,1.42.0,NaN,7.692012


In [35]:
df_wide = df.pivot_table(
    index=["referenceTime", "sourceId"],
    columns="elementId",
    values="value",
    aggfunc="first"
).reset_index()
df_wide

elementId,referenceTime,sourceId,air_temperature,sum(precipitation_amount PT1H)
0,2026-09-01 00:00:00+00:00,SN99950:0,3.7,0.0
1,2026-09-01 01:00:00+00:00,SN99950:0,3.5,0.0
2,2026-09-01 02:00:00+00:00,SN99950:0,3.6,0.0
3,2026-09-01 03:00:00+00:00,SN99950:0,3.6,0.0
4,2026-09-01 04:00:00+00:00,SN99950:0,3.4,0.0
...,...,...,...,...
715,2026-09-30 19:00:00+00:00,SN99950:0,8.3,2.3
716,2026-09-30 20:00:00+00:00,SN99950:0,8.3,2.6
717,2026-09-30 21:00:00+00:00,SN99950:0,8.5,0.1
718,2026-09-30 22:00:00+00:00,SN99950:0,8.7,0.0


In [20]:
params = {
    "sources": "SN99950", "elements": 'air_temperature',
    "referencetime": "2026-09-01/2026-10-01",
}
response = requests.get(url, params=params, auth=(CLIENT_ID, ""))
# response.raise_for_status()
data = response.json()
stations = pd.DataFrame(data['data'])
stations

,sourceId,referenceTime,observations
0,SN99950:0,2026-09-01T00:00:00.000Z,"[{'elementId': 'air_temperature', 'value': 3.7..."
1,SN99950:0,2026-09-01T00:10:00.000Z,"[{'elementId': 'air_temperature', 'value': 3.5..."
2,SN99950:0,2026-09-01T00:20:00.000Z,"[{'elementId': 'air_temperature', 'value': 3.3..."
3,SN99950:0,2026-09-01T00:30:00.000Z,"[{'elementId': 'air_temperature', 'value': 3.5..."
4,SN99950:0,2026-09-01T00:40:00.000Z,"[{'elementId': 'air_temperature', 'value': 3.5..."
...,...,...,...
4315,SN99950:0,2026-09-30T23:10:00.000Z,"[{'elementId': 'air_temperature', 'value': 8.9..."
4316,SN99950:0,2026-09-30T23:20:00.000Z,"[{'elementId': 'air_temperature', 'value': 8.8..."
4317,SN99950:0,2026-09-30T23:30:00.000Z,"[{'elementId': 'air_temperature', 'value': 8.9..."
4318,SN99950:0,2026-09-30T23:40:00.000Z,"[{'elementId': 'air_temperature', 'value': 8.9..."


In [25]:
stations['observations'][0]

[{'elementId': 'air_temperature',
  'value': 3.7,
  'unit': 'degC',
  'level': {'levelType': 'height_above_ground', 'unit': 'm', 'value': 2},
  'timeOffset': 'PT0H',
  'timeResolution': 'PT1H',
  'timeSeriesId': 0,
  'performanceCategory': 'C',
  'exposureCategory': '1',
  'qualityCode': 0},
 {'elementId': 'air_temperature',
  'value': 3.7,
  'unit': 'degC',
  'level': {'levelType': 'height_above_ground', 'unit': 'm', 'value': 2},
  'timeOffset': 'PT0H',
  'timeResolution': 'PT10M',
  'timeSeriesId': 0,
  'performanceCategory': 'C',
  'exposureCategory': '1',
  'qualityCode': 0}]

In [26]:
stations['observations'][1]

[{'elementId': 'air_temperature',
  'value': 3.5,
  'unit': 'degC',
  'level': {'levelType': 'height_above_ground', 'unit': 'm', 'value': 2},
  'timeOffset': 'PT0H',
  'timeResolution': 'PT10M',
  'timeSeriesId': 0,
  'performanceCategory': 'C',
  'exposureCategory': '1',
  'qualityCode': 0}]

In [27]:
stations['observations'][2]

[{'elementId': 'air_temperature',
  'value': 3.3,
  'unit': 'degC',
  'level': {'levelType': 'height_above_ground', 'unit': 'm', 'value': 2},
  'timeOffset': 'PT0H',
  'timeResolution': 'PT10M',
  'timeSeriesId': 0,
  'performanceCategory': 'C',
  'exposureCategory': '1',
  'qualityCode': 0}]

In [ ]:
pd.DataFrame(stations)['observations'].iloc[0]

In [ ]:
import requests

url = "https://frost.met.no/observations/availableTimeSeries/v0.jsonld"

params = {
    "sources": "SN47230"
}

response = requests.get(
    url,
    params=params,
    auth=(CLIENT_ID, "")
)

response.raise_for_status()

data = response.json()

print(data.keys())

In [ ]:
for item in data["data"]:
    print(
        "Source:", item.get("sourceId"),
        "| Element:", item.get("elementId"),
        "| Unit:", item.get("unit"),
        "| Resolution:", item.get("timeResolution"),
        "| From:", item.get("validFrom"),
        "| To:", item.get("validTo")
    )

In [ ]:
df.columns

In [ ]:
df['geometry'] = df['geometry'].apply(lambda x: x['coordinates'] if isinstance(x, dict) and 'coordinates' in x else None)

In [ ]:
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['geometry'].apply(lambda x: x[0] if x else None), df['geometry'].apply(lambda x: x[1] if x else None)), crs='EPSG:4326')

In [ ]:
gdf

In [ ]:
gdf.to_file("met_stations.geojson", driver='GeoJSON')